# SPLADE: эксперименты (MS MARCO)

Единая точка запуска: клацай клетки по порядку — всё остальное в `splade_lab/` (общий код).

Порядок:
1. **Setup** — окружение и устройство.
2. **Конфиг данных** — режим (`smoke`/`full`) и источники MS MARCO.
3. **Конфиги экспериментов** — две версии SPLADE, все гиперпараметры видны здесь.
4. **Данные** — скачиваются один раз, переиспользуются всеми версиями.
5. **Прогоны** — обучение + eval, артефакты в `outputs/<version>/<run_id>/`.
6. **Сравнение** — сводная таблица метрик всех прогонов.

По умолчанию `MODE = "smoke"`: маленький детерминированный сабсет, минуты.
`"full"` — полный MS MARCO: гигабайты трафика и часы обучения на A100.

## 1. Setup

In [1]:
# Первый запуск на новой машине (в активированном venv) — раскомментировать:
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2

import json
import torch

from splade_lab.config import merge_config
from splade_lab.train import pick_device

device = pick_device()
print(f"torch {torch.__version__} | устройство: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

Looking in indexes: https://pypi.yandex-team.ru/simple/
Note: you may need to restart the kernel to use updated packages.
torch 2.5.1+cu124 | устройство: cuda (NVIDIA A100 80GB PCIe)


## 2. Конфиг данных

`smoke` собирается из первых строк `triples.train.small` (тексты query/pos/neg уже в файле,
полная коллекция не нужна, ~10–30MB трафика). `full` — коллекция 8.8M пассажей + dev small
(6980 запросов) + срез triples. Обе версии экспериментов читают один и тот же каталог.

In [11]:
MODE = "full"  # "smoke" | "full"  (full — гигабайты и часы, см. предупреждение при подготовке)

DATA = {
    # Официальные файлы Microsoft. Если хост недоступен — зеркало:
    # https://msmarco.blob.core.windows.net/msmarcoranking/
    "urls": {
        "triples": "https://msmarco.z22.web.core.windows.net/msmarcoranking/triples.train.small.tar.gz",
        "collection": "https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz",
        "queries": "https://msmarco.z22.web.core.windows.net/msmarcoranking/queries.tar.gz",
        "qrels_dev": "https://msmarco.z22.web.core.windows.net/msmarcoranking/qrels.dev.small.tsv",
    },
    "data_dir": "data/msmarco",  # относительно корня репозитория
    "smoke": {
        "num_train_triples": 256,  # строк triples на обучение
        "num_eval_queries": 20,    # уникальных eval-запросов (не пересекаются с train)
        "num_corpus_docs": 1000,   # корпус: positives eval-запросов + дистракторы
    },
    "full": {
        "num_train_triples": 3_200_000,  # ~ max_steps * batch_size; -1 = весь файл (39.8M строк)
        "num_eval_queries": -1,          # -1 = все 6980 запросов dev small
    },
}

print(f"MODE = {MODE} | параметры данных: {DATA[MODE]}")

MODE = full | параметры данных: {'num_train_triples': 3200000, 'num_eval_queries': -1}


## 3. Конфиги экспериментов

Две версии из SPLADE v2 (arXiv:2109.10086), общий код — разные словари:
- **v1_splade_max** — общий MLM-энкодер для запросов и документов (SPLADE-max);
- **v2_splade_doc** — энкодер только для документов, запрос = бинарный мешок токенов (SPLADE-doc).

Новая версия = новая запись в `EXPERIMENTS` (переопределения поверх `BASE`). Код не трогать.

In [ ]:
# База: гиперпараметры статьи (distilbert, Adam 2e-5, warmup 6000, max_len 256).
# Отличие от статьи: 50k шагов и batch 64 (1xA100 против 4xV100 / batch 124 / 150k).
BASE = {
    "model": {
        "hf_model": "distilbert-base-uncased",
        "query_encoder": "mlm",      # mlm = SPLADE-max | bow = SPLADE-doc
        "max_len_query": 32,
        "max_len_doc": 256,
    },
    "train": {
        "seed": 4,
        "lr": 2e-5,
        "batch_size": 124,
        "max_steps": 100_000,
        "warmup_steps": 6_000,
        "flops_warmup_steps": 10_000,  # квадратичный разгон лямбд (SPLADE v2)
        "lambda_q": 3e-4,              # FLOPS-рег. запросов
        "lambda_d": 1e-4,              # FLOPS-рег. документов
        "log_every": 100,
    },
    "eval": {
        "batch_size_docs": 256,
        "batch_size_queries": 64,
        "batch_size_search": 64,
        "recall_ks": [10, 100, 1000],
    },
}

# Smoke: всё маленькое, схема артефактов та же.
SMOKE_OVERRIDES = {
    "model": {"max_len_doc": 128},
    "train": {"max_steps": 20, "batch_size": 8, "warmup_steps": 4,
              "flops_warmup_steps": 8, "log_every": 5},
    "eval": {"batch_size_docs": 32, "batch_size_queries": 32, "recall_ks": [10, 100]},
}

# Версии экспериментов: имя -> переопределения поверх BASE.
EXPERIMENTS = {
    "v1_splade_max": {},
    "v2_splade_doc": {
        "model": {"query_encoder": "bow"},
        "train": {
                    "lambda_q": 0.0,      # запрос фиксирован, рег. не нужна
                    "lambda_d": 3e-4,     # рег. только на документы
                    "max_steps": 50_000
                },
    },
}

def build_config(name: str) -> dict:
    cfg = merge_config(BASE, EXPERIMENTS[name])
    if MODE == "smoke":
        cfg = merge_config(cfg, SMOKE_OVERRIDES)
    cfg["version"], cfg["mode"] = name, MODE
    return cfg

for name in EXPERIMENTS:
    print(f"=== {name} (mode={MODE}) ===")
    print(json.dumps(build_config(name), indent=2, ensure_ascii=False))

=== v1_splade_max (mode=full) ===
{
  "model": {
    "hf_model": "distilbert-base-uncased",
    "query_encoder": "mlm",
    "max_len_query": 32,
    "max_len_doc": 256
  },
  "train": {
    "seed": 4,
    "lr": 2e-05,
    "batch_size": 64,
    "max_steps": 250000,
    "warmup_steps": 16000,
    "flops_warmup_steps": 20000,
    "lambda_q": 0.0003,
    "lambda_d": 0.0001,
    "log_every": 100
  },
  "eval": {
    "batch_size_docs": 256,
    "batch_size_queries": 64,
    "batch_size_search": 64,
    "recall_ks": [
      10,
      100,
      1000
    ]
  },
  "version": "v1_splade_max",
  "mode": "full"
}
=== v2_splade_doc (mode=full) ===
{
  "model": {
    "hf_model": "distilbert-base-uncased",
    "query_encoder": "bow",
    "max_len_query": 32,
    "max_len_doc": 256
  },
  "train": {
    "seed": 4,
    "lr": 2e-05,
    "batch_size": 64,
    "max_steps": 250000,
    "warmup_steps": 16000,
    "flops_warmup_steps": 20000,
    "lambda_q": 0.0,
    "lambda_d": 0.0003,
    "log_every": 100


## 4. Данные (скачиваются один раз)

In [16]:
from splade_lab import data as msdata

prepare = msdata.prepare_full if MODE == "full" else msdata.prepare_smoke
ds_dir = prepare(DATA)

print(f"\nКаталог: {ds_dir}")
print("Строк в файлах:", msdata.dataset_stats(ds_dir))

with open(ds_dir / "queries.tsv", encoding="utf-8") as f:
    print("\nПример запроса:  ", f.readline().strip()[:120])
with open(ds_dir / "collection.tsv", encoding="utf-8") as f:
    print("Пример пассажа:  ", f.readline().strip()[:120], "...")

[data] full уже готов: /home/uvuv/workspace/sandbox_03/data/msmarco/full

Каталог: /home/uvuv/workspace/sandbox_03/data/msmarco/full
Строк в файлах: {'collection.tsv': 8841823, 'queries.tsv': 6980, 'qrels.tsv': 7437, 'triples.tsv': 3200000}

Пример запроса:   2	 Androgen receptor define
Пример пассажа:   0	The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as s ...


## 5. Прогоны

Каждый прогон: обучение → сохранение весов → eval → `outputs/<version>/<run_id>/`
(`config.json`, `metrics.json`, `meta.json` с seed/commit/версиями пакетов, `model/`).

In [ ]:
from splade_lab.train import run_experiment

results = {}
for name in EXPERIMENTS:
    cfg = build_config(name)
    run_dir, metrics = run_experiment(cfg, DATA)
    results[name] = {"run_dir": str(run_dir), **metrics}

for name, res in results.items():
    print(f"{name}: mrr@10={res.get('mrr@10'):.4f} | {res['run_dir']}")

[run] v1_splade_max mode=full device=cuda -> /home/uvuv/workspace/sandbox_03/outputs/v1_splade_max/20260612-003154


load:triples: 0.00 строк [00:00, ? строк/s]

train:v1_splade_max:   0%|          | 0/250000 [00:00<?, ? шаг/s]

load:collection: 0.00 строк [00:00, ? строк/s]

encode:doc:   0%|          | 0/34539 [00:00<?, ?it/s]

...

search:gpu:   0%|          | 0/110 [00:00<?, ?it/s]

[run] готово: /home/uvuv/workspace/sandbox_03/outputs/v1_splade_max/20260612-003154
[run] метрики: {'mrr@10': 0.3231409469231819, 'recall@10': 0.588932664756447, 'recall@100': 0.8547397325692455, 'recall@1000': 0.9646489971346704, 'n_eval_queries': 6980, 'n_corpus_docs': 8841823, 'avg_nnz_doc': 147.95362619224565, 'avg_nnz_query': 28.976647564469914, 'final_train_loss': 0.001052217569667846, 'train_steps': 250000}
[run] v2_splade_doc mode=full device=cuda -> /home/uvuv/workspace/sandbox_03/outputs/v2_splade_doc/20260612-131628


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by NewConnectionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to establish a new connection: [Errno 13] Permission denied"))'), '(Request ID: d89b3028-c2e4-468c-a52c-bb89b595e700)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/tokenizer_config.json (Caused by NewConnectionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to establish a new connection: [Errno 13] Permission denied"))'), '(Request ID: 2b79b259-e4fb-453f-bd59-d7b380b19f04)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json


load:triples: 0.00 строк [00:00, ? строк/s]

train:v2_splade_doc:   0%|          | 0/250000 [00:00<?, ? шаг/s]

## 6. Сравнение версий (все прогоны из outputs/)

In [1]:
from splade_lab.compare import compare_runs

df = compare_runs()  # печатает таблицу и пишет outputs/comparison.csv
df

      version          run_id  mode  seed   mrr@10  recall@10  recall@100  recall@1000  avg_nnz_doc  avg_nnz_query  final_train_loss  train_steps  n_eval_queries  n_corpus_docs  duration_s
v1_splade_max 20260612-003154  full     4 0.323141   0.588933     0.85474     0.964649   147.953626      28.976648          0.001052       250000            6980        8841823     45874.6
v1_splade_max 20260612-002533 smoke    42 0.248175   0.550000     0.85000          NaN  5150.361000    1145.800000         34.342426           20              20           1000        10.5
v1_splade_max 20260612-002635 smoke     4 0.289008   0.650000     0.90000          NaN  4406.685000    1073.750000         28.770599           20              20           1000         2.5
v2_splade_doc 20260612-002543 smoke    42 0.879167   1.000000     1.00000          NaN   985.329000       7.450000          0.525460           20              20           1000         2.2
v2_splade_doc 20260612-002637 smoke     4 0.912500   1.

,version,run_id,mode,seed,mrr@10,recall@10,recall@100,recall@1000,avg_nnz_doc,avg_nnz_query,final_train_loss,train_steps,n_eval_queries,n_corpus_docs,duration_s
0,v1_splade_max,20260612-003154,full,4,0.323141,0.588933,0.85474,0.964649,147.953626,28.976648,0.001052,250000,6980,8841823,45874.6
1,v1_splade_max,20260612-002533,smoke,42,0.248175,0.550000,0.85000,NaN,5150.361000,1145.800000,34.342426,20,20,1000,10.5
2,v1_splade_max,20260612-002635,smoke,4,0.289008,0.650000,0.90000,NaN,4406.685000,1073.750000,28.770599,20,20,1000,2.5
3,v2_splade_doc,20260612-002543,smoke,42,0.879167,1.000000,1.00000,NaN,985.329000,7.450000,0.525460,20,20,1000,2.2
4,v2_splade_doc,20260612-002637,smoke,4,0.912500,1.000000,1.00000,NaN,826.187000,7.450000,0.549977,20,20,1000,2.1


## Как добавить версию / запустить full

**Новая версия:** добавь запись в `EXPERIMENTS` (клетка 3), например
`"v3_high_reg": {"train": {"lambda_d": 1e-3}}`, и перезапусти клетки 3 → 5 → 6.

**Full-прогон (не запускался):** поставь `MODE = "full"` в клетке 2 и пройди клетки
2 → 3 → 4 → 5 → 6 заново. Ресурсы: ~1GB трафика + ~6GB диска на данные, обучение 50k
шагов — часы на A100, кодирование 8.8M пассажей ~1–2 ч, поиск — десятки минут.

**Ориентиры качества (SPLADE v2, full):** SPLADE-max MRR@10 ≈ 0.340, SPLADE-doc ≈ 0.322;
здесь ожидаемо чуть ниже (50k шагов / batch 64). Smoke-метрики — только проверка пайплайна,
не качество.